In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shreyasparaj1/loan-prediction-dataset")

print("Path to dataset files:", path)

c:\Users\mrayy\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\mrayy\.cache\kagglehub\datasets\shreyasparaj1\loan-prediction-dataset\versions\1


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Pehle file ka naam check karte hain directory mein
files = os.listdir(path)
print("Files in dataset:", files)

# Agar file 'loan_data.csv' hai (aap file ka naam check kar lein)
# Toh is tarah load karein:
df = pd.read_csv(os.path.join(path, files[1]))  # files[1] ko apne file ke index se replace karein

Files in dataset: ['test_Y3wMUE5_7gLdaTN.csv', 'train_u6lujuX_CVtuZ9i.csv']


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# --- 1. Preprocessing aur Missing Values ---
# Credit History ko Mode (1.0) se fill karna sab se zaroori hai
df['Credit_History'] = df['Credit_History'].fillna(df['Credit_History'].mode()[0])
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mode()[0])
df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].median())

# Baqi categorical values ko bhi mode se fill kar dein
df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])
df['Married'] = df['Married'].fillna(df['Married'].mode()[0])
df['Dependents'] = df['Dependents'].fillna(df['Dependents'].mode()[0])
df['Self_Employed'] = df['Self_Employed'].fillna(df['Self_Employed'].mode()[0])

# --- 2. Smart Feature Engineering ---
# Total Income banayen
df['Total_Income'] = df['ApplicantIncome'] + df['CoapplicantIncome']

# EMI (Monthly Installment) banayen
# Formula: LoanAmount / Loan_Amount_Term
df['EMI'] = df['LoanAmount'] / df['Loan_Amount_Term']

# Balance Income (Ghar chalane ke liye kitne paise bachenge)
df['Balance_Income'] = df['Total_Income'] - (df['EMI'] * 1000) # 1000 se multiply kyunke loan k thousands mein hota hai

# Incomes ka log lein taake data normal ho jaye
df['Total_Income_log'] = np.log(df['Total_Income'])

# --- 3. Encoding ---
# Simple Mapping
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
df['Married'] = df['Married'].map({'Yes': 1, 'No': 0})
df['Education'] = df['Education'].map({'Graduate': 1, 'Not Graduate': 0})
df['Self_Employed'] = df['Self_Employed'].map({'Yes': 1, 'No': 0})
df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})
df['Dependents'] = df['Dependents'].replace('3+', 3).astype(int)

# One-Hot Encoding for Property Area
df = pd.get_dummies(df, columns=['Property_Area'], drop_first=True)

# --- 4. Feature Selection ---
# Fuzool columns (original incomes aur ID) ko nikal dein
X = df.drop(['Loan_ID', 'Loan_Status', 'ApplicantIncome', 'CoapplicantIncome', 'Total_Income'], axis=1)
y = df['Loan_Status']

# --- 5. Train-Test Split ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- 6. Random Forest Model ---
# Hum Random Forest use kar rahe hain kyunke yeh non-linear patterns behtar samajhta hai
model = RandomForestClassifier(n_estimators=100, max_depth=5, min_samples_leaf=10, random_state=42)
model.fit(X_train, y_train)

# --- 7. Final Score ---
ypred = model.predict(X_test)
print(f"Improved Accuracy: {accuracy_score(y_test, ypred) * 100:.2f}%")

Improved Accuracy: 78.05%
